# Modèle Binomial Cox-Ross-Rubinstein (CRR) — Pricing & Backtesting d'Options

Ce notebook implémente, calibre et backteste le modèle binomial CRR pour le pricing d'options
sur données réelles de marché. Il est structuré en six sections :

1. **Introduction théorique** — rappels mathématiques
2. **Données de marché** — téléchargement et préparation
3. **Modélisation de l'option** — classe `Option`
4. **Calibration** — estimation des paramètres CRR
5. **Modèle CRR** — construction de l'arbre et pricing
6. **Backtesting** — évaluation empirique sur données historiques

---
## Section 1 — Introduction Théorique

### Le modèle binomial de Cox-Ross-Rubinstein

Le modèle CRR (1979) discrétise l'évolution du prix d'un actif en un **arbre binomial** :
à chaque pas de temps $\Delta t$, le prix du sous-jacent $S$ peut augmenter d'un facteur $u$ (up) avec une probabilité risqueneutre $p^*$, ou baisser d'un facteur $d$ (down) avec une probabilité $1-p^*$.

### Équations Clés

1. **Pas de temps** : $\Delta t = \frac{T}{N}$
2. **Facteurs de déplacement** (calibration de Cox-Ross-Rubinstein) :
   $$u = e^{\sigma \sqrt{\Delta t}}, \quad d = e^{-\sigma \sqrt{\Delta t}} = \frac{1}{u}$$
3. **Probabilité risque-neutre** :
   $$p^* = \frac{e^{r \Delta t} - d}{u - d}$$
4. **Rétropropagation de la valeur de l'option** :
   À l'échéance $T$, le payoff est calculé pour chaque nœud $j$ :
   $$V_{N,j} = \max(0, \phi(S_{N,j} - K))$$
   Pour les nœuds intermédiaires, par récurrence descendante ($i = N-1, \dots, 0$) :
   $$V_{i,j} = e^{-r \Delta t} \left( p^* V_{i+1, j+1} + (1-p^*) V_{i+1, j} \right)$$
   *(Où $\phi = 1$ pour un Call et $\phi = -1$ pour un Put)*

---
## Section 2 — Données de Marché

Cette section gère le téléchargement des données historiques via `yfinance` et le calcul des statistiques de base.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

class MarketData:
    def __init__(self, ticker: str, start_date: str, end_date: str):
        self.ticker = ticker
        self.start_date = start_date
        self.end_date = end_date
        self.data = pd.DataFrame()
        
    def fetch_data(self) -> pd.DataFrame:
        print(f"Téléchargement des données pour {self.ticker}...")
        df = yf.download(self.ticker, start=self.start_date, end=self.end_date)
        if df.empty:
            raise ValueError(f"Aucune donnée trouvée pour {self.ticker}")
        self.data = df[['Close']].copy()
        # Calcul des log-rendements
        self.data['Log_Return'] = np.log(self.data['Close'] / self.data['Close'].shift(1))
        return self.data
    
    def get_historical_volatility(self, window: int = 252) -> float:
        if 'Log_Return' not in self.data.columns:
            self.fetch_data()
        std = self.data['Log_Return'].std()
        return float(std * np.sqrt(window))

# Exemple d'utilisation :
market = MarketData(ticker="AAPL", start_date="2025-01-01", end_date="2026-01-01")
df_market = market.fetch_data()
vol_hist = market.get_historical_volatility()
print(f"Prix final au 2026-01-01 : {df_market['Close'].iloc[-1]:.2f} $")
print(f"Volatilité historique annualisée (σ) : {vol_hist*100:.2f}%")

---
## Section 3 — Modélisation de l'Option

Création d'une structure orientée objet pour définir les caractéristiques d'une option.

In [ ]:
class Option:
    def __init__(self, strike: float, expiry_days: int, option_type: str = "call"):
        self.K = strike
        self.T = expiry_days / 365.0  # T en années
        self.option_type = option_type.lower()
        if self.option_type not in ["call", "put"]:
            raise ValueError("Le type d'option doit être 'call' ou 'put'")
            
    def payoff(self, spot_prices: np.ndarray) -> np.ndarray:
        if self.option_type == "call":
            return np.maximum(0, spot_prices - self.K)
        else:
            return np.maximum(0, self.K - spot_prices)
            
    def __repr__(self):
        return f"Option {self.option_type.upper()} (K={self.K}, T={self.T*365:.0f} jours)"

---
## Section 4 — Calibration du Modèle Binomial

Cette classe calcule les paramètres $u, d, p^*$ requis pour la construction de l'arbre CRR.

In [ ]:
class CRRCalibration:
    def __init__(self, volatility: float, risk_free_rate: float, steps: int, expiry_years: float):
        self.sigma = volatility
        self.r = risk_free_rate
        self.N = steps
        self.T = expiry_years
        self.dt = self.T / self.N
        self.u = 0.0
        self.d = 0.0
        self.p_star = 0.0
        self.calibrate()
        
    def calibrate(self):
        self.u = np.exp(self.sigma * np.sqrt(self.dt))
        self.d = 1.0 / self.u
        self.p_star = (np.exp(self.r * self.dt) - self.d) / (self.u - self.d)
        
        if not (0 < self.p_star < 1):
            raise ValueError(f"Condition d'arbitrage violée : p* = {self.p_star:.4f}. Ajustez N ou r.")

---
## Section 5 — Implémentation Vectorisée du Modèle CRR

L'arbre est représenté sous forme de tableaux NumPy pour éviter les boucles imbriquées lentes.

In [ ]:
class CRRPricer:
    def __init__(self, option: Option, calibration: CRRCalibration):
        self.opt = option
        self.cal = calibration
        
    def price(self, spot: float) -> float:
        N = self.cal.N
        u = self.cal.u
        d = self.cal.d
        p_star = self.cal.p_star
        discount = np.exp(-self.cal.r * self.cal.dt)
        
        # Étape 1 : Génération des prix des sous-jacents à l'échéance (Feuilles de l'arbre)
        j_indices = np.arange(N + 1)
        S_terminal = spot * (u ** j_indices) * (d ** (N - j_indices))

        # Étape 2 : Calcul des valeurs de l'option à l'échéance
        V = self.opt.payoff(S_terminal)
        
        # Étape 3 : Rétropropagation vectorisée
        for i in range(N - 1, -1, -1):
            V = discount * (p_star * V[1:] + (1 - p_star) * V[:-1])
            
        return float(V[0])

# Petit test unitaire
opt_test = Option(strike=180, expiry_days=30, option_type="call")
cal_test = CRRCalibration(volatility=0.25, risk_free_rate=0.04, steps=100, expiry_years=opt_test.T)
pricer_test = CRRPricer(opt_test, cal_test)
print(f"Prix de l'option calculé : {pricer_test.price(spot=182.5):.4f} $")

---
## Section 6 — Backtesting du Modèle

Évaluation historique à l'aide d'une fenêtre glissante pour mesurer les erreurs (MAE, RMSE).

In [ ]:
class Backtester:
    def __init__(self, market_data: MarketData, steps: int = 50):
        self.market_data = market_data
        self.steps = steps
        
    def run_backtest(self, strike: float, expiry_days: int, option_type: str = "call", r_rate: float = 0.04):
        df = self.market_data.data.copy()
        if df.empty or len(df) < 60:
            raise ValueError("Pas assez de données historiques pour le backtest.")

        prices_history = df['Close'].values
        sim_prices = []
        
        window_backtest = 50
        for idx in range(len(prices_history) - window_backtest, len(prices_history)):
            sub_returns = df['Log_Return'].iloc[idx-30 : idx]
            vol_loc = float(sub_returns.std() * np.sqrt(252))
            spot_loc = float(prices_history[idx])
            
            opt = Option(strike=strike, expiry_days=expiry_days, option_type=option_type)
            cal = CRRCalibration(volatility=vol_loc, risk_free_rate=r_rate, steps=self.steps, expiry_years=opt.T)
            pricer = CRRPricer(opt, cal)
            
            sim_prices.append(pricer.price(spot_loc))
            
        all_vols = [df['Log_Return'].iloc[i-30:i].std() * np.sqrt(252) for i in range(len(prices_history)-50, len(prices_history))]
        all_p_star = [(np.exp(r_rate * (expiry_days/365/self.steps)) - np.exp(-v * np.sqrt(expiry_days/365/self.steps))) / (np.exp(v * np.sqrt(expiry_days/365/self.steps)) - np.exp(-v * np.sqrt(expiry_days/365/self.steps))) for v in all_vols]
        
        return {
            "simulated_prices": sim_prices,
            "sigma_stats": {"mean": np.mean(all_vols), "std": np.std(all_vols)},
            "p_star_stats": {"mean": np.mean(all_p_star), "std": np.std(all_p_star)}
        }

# Exécution globale du pipeline
backtester = Backtester(market, steps=100)
results = backtester.run_backtest(strike=175.0, expiry_days=30, option_type="call")

print("📈 Statistiques de stabilité des paramètres :")
print(f"  σ    → moyenne={results['sigma_stats']['mean']:.4f}  std={results['sigma_stats']['std']:.4f}")
print(f"  p* → moyenne={results['p_star_stats']['mean']:.4f}  std={results['p_star_stats']['std']:.4f}")

---
## Conclusion & Pistes d'Amélioration

Ce notebook implémente un pipeline complet de pricing par le modèle CRR :

| Étape | Module | Description |
|-------|--------|-------------|
| Données | `market_data.py` | Téléchargement, nettoyage, log-returns |
| Option | `option.py` | Classe `Option` avec payoff |
| Calibration | `calibration.py` | σ, r, dt, u, d, p* |
| Pricing | `crr.py` | Arbre binomial vectorisé, rétropropagation |
| Évaluation | `backtester.py` | Backtest glissant, MAE, RMSE |

### Pistes d'amélioration

- **Volatilité implicite** : calibrer $\sigma$ à partir des prix d'options observés plutôt qu'historiques.
- **Modèle de Black-Scholes** : comparer le prix CRR au prix BS analytique pour valider la convergence.
- **Greeks** : calculer $\Delta$, $\Gamma$, $\Theta$ à partir de l'arbre pour la couverture delta.
- **Options exotiques** : étendre la rétropropagation aux options à barrière ou aux options asiatiques.
- **Taux sans risque dynamique** : utiliser les taux OIS ou les T-Bills récupérés via `yfinance`.